# The 401k / passive-bid fingerprint — token-free (Colab)

You can't see Fidelity's actual trades (not disclosed). But the mechanical 401k/target-date-fund flow
leaves a **calendar fingerprint**, and that we CAN read from price alone. Three tests, each mapped to the
question:
1. **Turn-of-the-Month (TOM):** is there an outsized bid on the last day + first 3 days of each month
   (payroll-drip + rebalance window)? How much of the market's total return lives there?
2. **The bear's test — FLOOR or CONFIRM:** does the TOM bid work in DOWN markets (a floor) or only in UP
   markets (just confirming the cleared price)? Split by 200-day regime. *This is the debate point.*
3. **TDF rebalance drag:** when stocks crush bonds in a month, target-date funds must SELL stocks into
   month-end. Does a big equity-over-bond month predict weak LAST-3-days (mechanical sell)?

Run top-to-bottom. yfinance with Stooq fallback for SPX; bond leg (AGG) optional.


In [ ]:
import sys, subprocess
def _pip(p): subprocess.run([sys.executable,'-m','pip','install','-q',p])
try:
    import pandas as pd, numpy as np
except Exception:
    _pip('pandas'); _pip('numpy'); import pandas as pd, numpy as np

def load_yf(sym, start='1993-01-01'):
    try:
        import yfinance as yf
    except Exception:
        _pip('yfinance'); import yfinance as yf
    df = yf.download(sym, start=start, progress=False, auto_adjust=True)
    if len(df):
        s = df['Close']; s = s.iloc[:,0] if hasattr(s,'columns') else s
        return pd.Series(np.asarray(s).ravel(), index=df.index, name=sym).dropna()
    return None

def load_spx():
    s = None
    try: s = load_yf('^GSPC')
    except Exception as e: print('yf SPX failed:', e)
    if s is not None and len(s): return s
    import urllib.request, io
    raw = urllib.request.urlopen('https://stooq.com/q/d/l/?s=^spx&i=d', timeout=30).read().decode()
    df = pd.read_csv(io.StringIO(raw), parse_dates=['Date']).set_index('Date').sort_index()
    return df['Close'].rename('spx').dropna()

spx = load_spx()
try: bond = load_yf('AGG')   # US aggregate bond ETF (2003+); TDF rebalance leg
except Exception: bond = None
print('SPX bars', len(spx), spx.index[0].date(), '->', spx.index[-1].date(),
      '| bond', (len(bond) if bond is not None else 'none'))


## Tag each trading day's position within its month


In [ ]:
d = pd.DataFrame({'px': spx}); d['ret'] = d['px'].pct_change()
d['ym'] = d.index.to_period('M')
g = d.groupby('ym')
d['dom_start'] = g.cumcount() + 1                      # 1 = first trading day of month
d['dom_end']   = g['px'].transform('size') - g.cumcount() - 1   # 0 = last trading day
d['sma200'] = d['px'].rolling(200).mean()
d['bull'] = d['px'] > d['sma200']
# TOM window = first 3 days + last day of month (straddles the turn in aggregate)
d['TOM'] = (d['dom_start'] <= 3) | (d['dom_end'] == 0)
d = d.dropna(subset=['ret'])


## TEST 1 — Turn-of-the-Month effect


In [ ]:
def stats(mask,label):
    r = d.loc[mask,'ret']
    return dict(window=label, days=int(len(r)), avg_bp=round(1e4*r.mean(),2),
               win=f'{100*(r>0).mean():.0f}%', ann=f'{100*r.mean()*252:.1f}%')
import pandas as pd
tom = d['TOM']
print(pd.DataFrame([stats(tom,'TOM (1st3+last)'), stats(~tom,'rest of month'),
                    stats(d.index==d.index,'ALL days')]).to_string(index=False))
# how much of TOTAL cumulative return lives in TOM days
tot = (1+d['ret']).prod() - 1
tomret = (1+d.loc[tom,'ret']).prod() - 1
print(f'\nTOM is {100*tom.mean():.0f}% of trading days but holds',
      f'{100*tomret/tot:.0f}% of the total cumulative return.')
print('If TOM >> its day-share, a mechanical month-turn bid is real.')


## TEST 2 — the BEAR's test: FLOOR or CONFIRM? (TOM split by 200-day regime)


In [ ]:
print('Does the month-turn bid work in DOWN markets (floor) or only UP (confirm)?')
print(pd.DataFrame([
    stats(tom & d['bull'],  'TOM | ABOVE 200d (bull)'),
    stats(tom & ~d['bull'], 'TOM | BELOW 200d (bear)'),
    stats(~tom & d['bull'], 'rest | bull'),
    stats(~tom & ~d['bull'],'rest | bear'),
]).to_string(index=False))
print('\nBear wins if TOM is strong ABOVE the 200d but weak/negative BELOW it:')
print('the bid CONFIRMS the cleared price in an uptrend, but does NOT floor a downtrend.')


## TEST 3 — TDF rebalance drag (needs bond leg AGG, 2003+)


In [ ]:
if bond is None or len(bond) < 250:
    print('No bond series (AGG) available — skipping TDF-rebalance test.')
else:
    m = pd.DataFrame({'spx': spx, 'agg': bond}).dropna()
    m['ym'] = m.index.to_period('M')
    # monthly equity-minus-bond spread, and the last-3-day equity return within each month
    mo = m.groupby('ym').apply(lambda x: pd.Series({
        'spread': x['spx'].iloc[-1]/x['spx'].iloc[0] - x['agg'].iloc[-1]/x['agg'].iloc[0],
        'last3': x['spx'].iloc[-1]/x['spx'].iloc[-4] - 1 if len(x) >= 4 else np.nan,
    })).dropna()
    c = mo['spread'].corr(mo['last3'])
    # split: big equity-outperformance months vs the rest
    hi = mo[mo['spread'] > mo['spread'].quantile(0.75)]
    lo = mo[mo['spread'] < mo['spread'].quantile(0.25)]
    print(f'corr(month equity-over-bond spread, last-3-day equity return) = {c:+.2f}')
    print(f'  months stocks CRUSHED bonds (top 25%): avg last-3-day = {1e4*hi["last3"].mean():+.1f} bp')
    print(f'  months stocks LAGGED bonds (bot 25%):  avg last-3-day = {1e4*lo["last3"].mean():+.1f} bp')
    print('\nNegative corr / weaker last-3-days after big equity months = mechanical rebalance SELL',
          'into month-end (TDFs trimming stocks to target). That flow is contrarian, not a floor.')


### How to read / caveats
- **This reads a FINGERPRINT, not Fidelity's trades.** A strong, regime-dependent TOM pattern is
  consistent with a mechanical bid; it does not *prove* 401k causation (could be other month-end flows).
- **Goodhart warning:** the turn-of-month effect is DECADES old and widely published — likely partly
  arbitraged away. Check whether it has DECAYED: rerun Test 1 on just the last ~8 years (add
  `d = d[d.index >= '2018-01-01']` before the tests) and see if the edge shrank. A known edge everyone
  trades is a thinner edge.
- **The payoff is Test 2.** If the bid only works above the 200-day, you've *confirmed the bear*: the
  passive/401k bid extends trend but doesn't backstop a drawdown — exactly 'confirms the cleared price,
  does not provide a floor.' That's a vault-grade answer either way it breaks.
- Paste the three tables back to me and we'll write the verdict into the bull/bear ledger.
